# Week 1: Linopy

https://linopy.readthedocs.io/en/latest/


In [ ]:
import linopy
import pandas as pd
import xarray as xr

In [ ]:
m = linopy.Model()

power_plant_types = pd.Index(["coal", "gas"], name="pp_types")
# costs = pd.DataFrame({"coal": 3, "gas": 4}, index=power_plant_types)
costs = xr.DataArray([3, 4], dims=["pp_types"], coords={"pp_types": ["coal", "gas"]})
# costs = xr.DataArray([3, 4], coords=[power_plant_types])
lower = xr.DataArray([50, 100], coords=[power_plant_types])
upper = xr.DataArray([300, 400], coords=[power_plant_types])
demand = 500

x = m.add_variables(
    lower=0,
    coords=[power_plant_types],
    name="power_generation",
)
m.add_objective(x * costs, overwrite=True)

min_constraint = m.add_constraints(x >= lower, name="Min Generation")
max_constraint = m.add_constraints(x <= upper, name="Max Generation")
balancing = m.add_constraints(x.sum() == demand, name="Suppy must equal Demand")

m.solve(solver_name="highs", output_flag=False)

In [ ]:
sol = m.solution.to_dataframe()
sol

In [ ]:
costs

In [ ]:
m.to_file("test.lp")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coal_opt = float(sol.loc["coal", "power_generation"])
gas_opt = float(sol.loc["gas", "power_generation"])

coal_bounds = (float(lower.sel(pp_types="coal")), float(upper.sel(pp_types="coal")))
gas_bounds = (float(lower.sel(pp_types="gas")), float(upper.sel(pp_types="gas")))
cost_coal = float(costs.sel(pp_types="coal"))
cost_gas = float(costs.sel(pp_types="gas"))

fig, ax = plt.subplots(figsize=(8, 8))

# balance line: coal + gas = demand
coal_line = np.linspace(0, demand, 200)
ax.plot(coal_line, demand - coal_line, color="blue")

# generator bounds
ax.axvline(coal_bounds[0], color="red")
ax.axvline(coal_bounds[1], color="red")
ax.axhline(gas_bounds[0], color="green")
ax.axhline(gas_bounds[1], color="green")

# feasible segment of the balance line
coal_feas_lo = max(coal_bounds[0], demand - gas_bounds[1])
coal_feas_hi = min(coal_bounds[1], demand - gas_bounds[0])
coal_feas = np.linspace(coal_feas_lo, coal_feas_hi, 200)
ax.plot(coal_feas, demand - coal_feas, color="gray", linewidth=4)

# corner points of the feasible segment, annotated with their objective value
for cx in (coal_feas_lo, coal_feas_hi):
    cy = demand - cx
    j = cost_coal * cx + cost_gas * cy
    ax.scatter([cx], [cy], color="gray", s=100, zorder=5)
    is_opt = np.isclose(cx, coal_opt) and np.isclose(cy, gas_opt)
    label = f"$J = {cost_coal:.0f}({cx:.0f}) + {cost_gas:.0f}({cy:.0f}) = {j:.0f}$"
    if is_opt:
        label = "Optimal solution\n" + label
    dx, dy = (30, -40) if cy > gas_opt else (-30, 40)
    ax.annotate(
        label,
        xy=(cx, cy),
        xytext=(cx + dx, cy + dy),
        ha="center",
        bbox=dict(boxstyle="round", fc="white", ec="black"),
    )

ax.set_xlim(0, demand)
ax.set_ylim(0, demand)
ax.set_xlabel(r"$P_{G1}$: Power from coal power plant [MW]")
ax.set_ylabel(r"$P_{G2}$: Power from gas power plant [MW]")
ax.grid(True)
plt.show()

In [ ]:
xr.DataArray([3, 4], dims=["power plants"], coords={"power plants": ["gas", "coal"]})

In [ ]:
power_plant_types

In [ ]:
costs

In [ ]:
m.objective